In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Application Message

### CQ2.01u - Which Application Message encodes which SOSA Activity?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?message ?activity ?member ?FOI ?property WHERE {
  ?message a mqtt:ApplicationMessage ;
          ?applicationMessageEncodesActivity ?activity .
  ?activity a mqtt:Activity .
  OPTIONAL {?activity sosa:hasFeatureOfInterest ?FOI}
  OPTIONAL {?activity sosa:hasMember ?member}
  OPTIONAL {?member sosa:observedProperty ?property}
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ2.02u - Which fields are encoded in the Application Message?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?fields WHERE {
  ?message a mqtt:ApplicationMessage ;
          mqtt:hasEncodedFields ?fields .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ2.03u - What content, content type, encoding and delimiter does an Application Message have?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?content ?type ?encoding ?delimiter WHERE {
  ?message a mqtt:ApplicationMessage ;
  OPTIONAL { ?message mqtt:hasApplicationContent ?content }
  OPTIONAL { ?message mqtt:hasApplicationContentType ?type }
  OPTIONAL { ?message mqtt:hasApplicationEncoding ?encoding }
  OPTIONAL { ?message mqtt:hasApplicationContentDelimiter ?delimiter }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ2.04u - Who has send which Application Messages?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?message ?payload ?packet ?client ?broker WHERE {
  ?message a mqtt:ApplicationMessage ;
           mqtt:isApplicationMessageOf ?payload .
  ?payload mqtt:isPublishPayloadOf ?packet .
  OPTIONAL { ?client mqtt:sendsControlPacket ?packet . ?client a mqtt:Client }
  OPTIONAL { ?broker mqtt:sendsControlPacket ?packet . ?broker a mqtt:Broker }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short